# camo-eval Colab Demo

This notebook exercises the lightweight camouflage evaluation path: detection, PR diagnostics, region/boundary scores, perceptual surrogates, generation-distance surrogates, and clutter/difficulty diagnostics. It intentionally avoids large benchmark datasets and pretrained model weights.

## 1. Install

Clone the repository so the bundled demo data is available, then install `camo-eval` in editable mode.

In [ ]:
!test -d CamoGED || git clone --depth 1 https://github.com/MichaelCSHN/CamoGED.git
!pip install -e CamoGED/camo-eval[full]

## 2. Single-pair metrics

In [ ]:
import numpy as np
from camo_eval import (
    dists, e_measure, f_measure, iou, dice, lpips, mae, ms_ssim,
    precision, recall, precision_recall_curve, s_measure, ssim,
    weighted_f_measure,
)

pred = np.array([[0, 255], [255, 0]], dtype=np.uint8)
gt = np.array([[0, 255], [255, 0]], dtype=np.uint8)

print('MAE:', mae(pred, gt))
print('Fw:', weighted_f_measure(pred, gt))
print('Sm:', s_measure(pred, gt))
print('Em:', e_measure(pred, gt))
print('F:', f_measure(pred, gt))
print('Precision:', precision(pred, gt))
print('Recall:', recall(pred, gt))
print('IoU:', iou(pred, gt))
print('Dice:', dice(pred, gt))
print('SSIM:', ssim(pred, gt))
print('MS-SSIM:', ms_ssim(pred, gt))
print('LPIPS_lite:', lpips(pred, gt))
print('DISTS_lite:', dists(pred, gt))


## 3. Batch evaluation

In [ ]:
from pathlib import Path
from camo_eval import evaluate, to_markdown

pred_dir = Path('pred_demo')
gt_dir = Path('gt_demo')
pred_dir.mkdir(exist_ok=True)
gt_dir.mkdir(exist_ok=True)

np.save(pred_dir / 'sample.npy', pred)
np.save(gt_dir / 'sample.npy', gt)

results = evaluate(
    pred_dir,
    gt_dir,
    ['mae', 'fw', 'sm', 'em', 'f', 'precision', 'recall', 'iou', 'dice', 'ssim', 'ms_ssim', 'lpips_lite', 'dists_lite'],
)
print(to_markdown(results))


## 4. Bundled repository demo dataset

In [ ]:
import json
from pathlib import Path

demo_root = Path('CamoGED/camo-eval/demo_data/cod_sota_masks')
manifest = json.loads((demo_root / 'manifest.json').read_text())
demo_metrics = manifest['metrics'] + ['lpips_lite', 'dists_lite']
demo_results = evaluate(
    demo_root / manifest['pred_dir'],
    demo_root / manifest['gt_dir'],
    demo_metrics,
)
print(to_markdown(demo_results))

## 5. Visualize mask, error map, and PR curve

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from camo_eval.visualization import error_map, mask_overlay

sample_pred = demo_root / manifest['pred_dir'] / 'sample1.png'
sample_gt = demo_root / manifest['gt_dir'] / 'sample1.png'
pred_img = np.asarray(Image.open(sample_pred))
gt_img = np.asarray(Image.open(sample_gt))
curve = precision_recall_curve(pred_img, gt_img)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(mask_overlay(pred_img, gt_img))
axes[0].set_title('mask overlay')
axes[1].imshow(error_map(pred_img, gt_img))
axes[1].set_title('error map')
axes[2].plot(curve['recall'], curve['precision'])
axes[2].set_title('PR curve')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].grid(alpha=0.3)
for ax in axes[:2]:
    ax.axis('off')
plt.tight_layout()


## 6. Lightweight generation-distance and clutter diagnostics

In [ ]:
from camo_eval import (
    camouflage_difficulty, edge_density, feature_congestion, fid, kid,
    subband_entropy,
)

sample_scene = demo_root / 'images' / 'sample1.png'
scene_img = np.asarray(Image.open(sample_scene))

print('FID_lite:', fid(str(demo_root / manifest['gt_dir']), str(demo_root / manifest['pred_dir'])))
print('KID_lite:', kid(str(demo_root / manifest['gt_dir']), str(demo_root / manifest['pred_dir'])))
print('edge_density:', edge_density(scene_img))
print('subband_entropy:', subband_entropy(scene_img))
print('feature_congestion:', feature_congestion(scene_img))
print('camouflage_difficulty:', camouflage_difficulty(scene_img, gt_img))


## 7. Protocol-aware reporting

In [ ]:
from camo_eval import EvaluationContext, EvaluationReport

context = EvaluationContext(
    observer='model',
    channel='rgb',
    task='image-cod',
    protocol='repository demo bundle',
)
report = EvaluationReport(context=context, metrics=demo_results.rows[0])
print(report.to_json())
print(report.to_markdown())